# De Novo Drug Discovery with Transformer Models for SMILES Generation

## Overview

This notebook provides a comprehensive end-to-end pipeline for generating novel drug-like molecules using transformer-based deep learning models. We'll work with SMILES (Simplified Molecular Input Line Entry System) representations - a compact string notation for describing molecular structures.

### What You'll Learn:

1. **Molecular Representations**: Understanding SMILES notation and its role in computational chemistry
2. **Data Preparation**: Downloading and processing large-scale molecular databases  
3. **Visualization**: Converting SMILES to 2D molecular structures using RDKit
4. **Tokenization**: Breaking down SMILES strings into learnable units
5. **Transformer Architecture**: Building a state-of-the-art generative model
6. **Next-Token Prediction**: Training the model to learn molecular grammar
7. **De Novo Generation**: Creating novel, chemically valid molecules
8. **Evaluation**: Assessing validity, uniqueness, and drug-likeness

### Applications:

- **Drug Discovery**: Generate novel molecular scaffolds for lead optimization
- **Chemical Space Exploration**: Sample diverse regions of chemical space
- **Property Optimization**: Design molecules with desired characteristics
- **Virtual Screening**: Expand compound libraries for computational screening

## 1. Environment Setup and Package Installation

First, we'll install all necessary packages for molecular manipulation, visualization, and deep learning.

In [1]:
# Install required packages
import sys

print("Installing required packages...")
print("This may take several minutes on first run.\n")

# Core packages
!{sys.executable} -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Chemistry and molecular packages
!{sys.executable} -m pip install -q rdkit

# Data and utilities
!{sys.executable} -m pip install -q pandas numpy matplotlib seaborn
!{sys.executable} -m pip install -q scikit-learn
!{sys.executable} -m pip install -q tqdm  # Progress bars
!{sys.executable} -m pip install -q requests  # For downloading datasets

print("\n✓ Installation complete!")

Installing required packages...
This may take several minutes on first run.



ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^



✓ Installation complete!


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
# Import all necessary libraries
import os
import random
import requests
import gzip
import io
from pathlib import Path
from collections import Counter, defaultdict
import re

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Chemistry
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, Lipinski, QED
from rdkit.Chem.Draw import IPythonConsole
from rdkit import RDLogger

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Progress tracking
from tqdm.auto import tqdm

# Suppress RDKit warnings
RDLogger.DisableLog('rdApp.*')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")


## 2. Data Download and Preparation

We'll download the MOSES dataset - a benchmarking platform for molecular generation. MOSES contains drug-like molecules from the ZINC database. **The notebook handles all data download and extraction automatically - no pre-existing files required!**


In [ ]:
# Create data directory
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

print("📥 Downloading SMILES Dataset...")
print("Source: MOSES - Molecular Sets (Benchmarking Platform)\\n")

def download_moses_dataset():
    """
    Download MOSES training dataset. This function handles everything automatically:
    - Downloads from GitHub if available
    - Falls back to curated sample if download fails
    - No manual setup required!
    """
    smiles_file = data_dir / "drug_like_smiles.txt"
    
    if smiles_file.exists():
        print(f"✓ Dataset already exists at {smiles_file}")
        return smiles_file
    
    try:
        # Download MOSES training set (1.9M molecules)
        url = "https://raw.githubusercontent.com/molecularsets/moses/master/data/train.csv"
        print(f"Downloading from {url}...")
        print("(This may take a few minutes)\\n")
        
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        # Parse CSV
        df = pd.read_csv(io.StringIO(response.text))
        smiles_list = df['SMILES'].tolist() if 'SMILES' in df.columns else df.iloc[:, 0].tolist()
        
        # Save
        with open(smiles_file, 'w') as f:
            f.write('\\n'.join(smiles_list))
        
        print(f"✓ Downloaded {len(smiles_list):,} SMILES strings!")
        return smiles_file
        
    except Exception as e:
        print(f"⚠️  Download failed: {e}")
        print("📝 Generating curated sample dataset...\\n")
        
        # Fallback: Use known drug molecules
        sample_smiles = [
            "CC(C)Cc1ccc(cc1)C(C)C(O)=O",  # Ibuprofen
            "CC(=O)Oc1ccccc1C(=O)O",  # Aspirin  
            "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # Caffeine
            "CC(C)NCC(COc1ccccc1)O",  # Propranolol
            "CN1CCC23C4C1CC5=C2C(=C(C=C5)O)OC3C(C=C4)O",  # Morphine
            "COc1ccc2nc(sc2c1)S(=O)(=O)N",  # Sulfamethoxazole
            "Cc1ccc(cc1)S(=O)(=O)N",  # Toluenesulfonamide
            "c1ccc2c(c1)ccc3c2ccc4c3cccc4",  # Anthracene
        ] * 1000  # Repeat for training
        
        with open(smiles_file, 'w') as f:
            f.write('\\n'.join(sample_smiles))
        
        print(f"✓ Generated {len(sample_smiles):,} sample molecules")
        return smiles_file

smiles_file = download_moses_dataset()

# Load the data
with open(smiles_file, 'r') as f:
    raw_smiles = [line.strip() for line in f if line.strip()]

print(f"\\n📊 Loaded {len(raw_smiles):,} SMILES strings")
print(f"\\n🔬 Sample molecules:")
for i, smi in enumerate(raw_smiles[:5], 1):
    print(f"  {i}. {smi}")


## 3. Molecular Visualization with RDKit

One of the most powerful aspects of SMILES is the ability to convert them into 2D/3D molecular structures. Let's visualize some molecules!


In [ ]:
# Visualize sample molecules
def visualize_molecules(smiles_list, n_mols=8, mols_per_row=4):
    """Convert SMILES to 2D molecular structures and display them."""
    selected = random.sample(smiles_list, min(n_mols, len(smiles_list)))
    mols = []
    legends = []
    
    for smi in selected:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols.append(mol)
            # Calculate properties
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            qed_score = QED.qed(mol)
            legends.append(f"MW:{mw:.1f} LogP:{logp:.2f}\\nQED:{qed_score:.2f}")
    
    # Draw grid
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(250, 250),
        legends=legends
    )
    return img

print("🔬 Sample Drug-like Molecules from Dataset\\n")
img = visualize_molecules(raw_smiles[:100], n_mols=8, mols_per_row=4)
display(img)

print("\\n💡 Key molecular properties:")
print("  • MW = Molecular Weight (Da)")
print("  • LogP = Lipophilicity (water/octanol partition)")  
print("  • QED = Quantitative Estimate of Drug-likeness (0-1, higher is better)")


## 4. SMILES Tokenization

To train a transformer model, we need to break SMILES strings into tokens. SMILES has a specific grammar with atoms, bonds, branches, and ring closures.


In [ ]:
class SMILESTokenizer:
    """
    Tokenizer for SMILES strings.
    Breaks SMILES into meaningful chemical tokens.
    """
    
    def __init__(self):
        # Pattern matches: Br, Cl, atoms in brackets, elements, special chars
        pattern = r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"
        self.regex = re.compile(pattern)
        
        # Special tokens
        self.PAD = '<PAD>'
        self.START = '<START>'
        self.END = '<END>'
        self.UNK = '<UNK>'
        
        self.token2idx = {}
        self.idx2token = {}
        self.vocab_size = 0
    
    def tokenize(self, smiles):
        """Split SMILES into tokens."""
        return self.regex.findall(smiles)
    
    def build_vocab(self, smiles_list):
        """Build vocabulary from SMILES list."""
        token_freq = Counter()
        
        print("Building vocabulary...")
        for smi in tqdm(smiles_list[:10000], desc="Tokenizing"):  # Sample for speed
            tokens = self.tokenize(smi)
            token_freq.update(tokens)
        
        # Add special tokens
        for token in [self.PAD, self.START, self.END, self.UNK]:
            self.token2idx[token] = len(self.token2idx)
        
        # Add regular tokens
        for token, _ in token_freq.most_common():
            if token not in self.token2idx:
                self.token2idx[token] = len(self.token2idx)
        
        self.idx2token = {idx: token for token, idx in self.token2idx.items()}
        self.vocab_size = len(self.token2idx)
        
        print(f"\n✓ Vocabulary: {self.vocab_size} unique tokens")
        print(f"  Top 10 tokens: {list(token_freq.most_common(10))}")
        
        return token_freq

# Build tokenizer
tokenizer = SMILESTokenizer()
token_freq = tokenizer.build_vocab(raw_smiles)

# Demo tokenization
example = raw_smiles[0]
tokens = tokenizer.tokenize(example)
print(f"\n📝 Tokenization Example:")
print(f"  SMILES: {example}")
print(f"  Tokens: {tokens}")
print(f"  Count: {len(tokens)} tokens")


## 5. SMILES Validation and Filtering

This notebook provides the foundation for SMILES-based molecular generation. To complete the pipeline, you would add:

### 5. Transformer Architecture
- **Positional Encoding**: Add position information to tokens
- **Multi-Head Self-Attention**: Learn molecular patterns
- **Feed-Forward Networks**: Transform representations
- **Causal Masking**: Enable autoregressive generation

### 6. Training
- **Next-Token Prediction**: Learn to predict each token given previous ones
- **Adam Optimizer**: With learning rate warmup
- **Validation**: Monitor on held-out molecules

### 7. Generation & Evaluation
- **Sampling Strategies**: Greedy, temperature, top-k, top-p
- **Validity**: Chemical validity via RDKit
- **Uniqueness**: Distinct generated molecules
- **Novelty**: Not in training set
- **Drug-likeness**: QED score

### Key Learning Points So Far

✅ **End-to-end data handling** - Automatic download and fallback  
✅ **SMILES representation** - Compact molecular notation  
✅ **RDKit visualization** - 2D molecular structures  
✅ **Tokenization** - Breaking SMILES into learnable units  
✅ **Property calculation** - MW, LogP, QED

### Resources

- **Papers**: \"Attention Is All You Need\" (Vaswani et al., 2017)
- **Benchmarks**: MOSES (molecularsets/moses on GitHub)
- **Libraries**: RDKit, PyTorch, transformers
- **Datasets**: ChEMBL, ZINC15, PubChem

🎓 **This notebook demonstrates the foundational concepts needed for transformer-based molecular generation!**


In [ ]:
def validate_smiles(smiles_list):
    """
    Validate SMILES and apply drug-likeness filters (Lipinski's Rule of Five).
    Filters: MW 150-500, LogP -2 to 5, HBD ≤5, HBA ≤10
    """
    valid_smiles = []
    
    for smi in tqdm(smiles_list[:5000], desc="Validating"):  # Limit for speed
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        
        # Calculate properties
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        
        # Apply filters
        if 150 <= mw <= 500 and -2 <= logp <= 5 and hbd <= 5 and hba <= 10:
            valid_smiles.append(Chem.MolToSmiles(mol))
    
    # Remove duplicates
    unique = list(set(valid_smiles))
    
    print(f"\n✓ Validation complete:")
    print(f"  Original: {len(smiles_list[:5000]):,}")
    print(f"  Valid & drug-like: {len(valid_smiles):,}")
    print(f"  Unique: {len(unique):,}")
    
    return unique

valid_smiles = validate_smiles(raw_smiles)
print(f"\n📊 Final dataset: {len(valid_smiles):,} molecules")


## 6. Dataset Preparation

Create PyTorch datasets and dataloaders for training.


In [ ]:
class SMILESDataset(Dataset):
    """PyTorch Dataset for SMILES strings."""
    
    def __init__(self, smiles_list, tokenizer, max_length=100):
        self.smiles = smiles_list
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.smiles)
    
    def __getitem__(self, idx):
        smi = self.smiles[idx]
        tokens = self.tokenizer.tokenize(smi)
        
        # Add special tokens
        tokens = [self.tokenizer.START] + tokens + [self.tokenizer.END]
        
        # Truncate if needed
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        
        # Convert to indices
        indices = [self.tokenizer.token2idx.get(t, self.tokenizer.token2idx[self.tokenizer.UNK]) 
                   for t in tokens]
        
        return torch.tensor(indices, dtype=torch.long)

def collate_fn(batch, pad_idx):
    """Pad batch to same length."""
    batch_padded = pad_sequence(batch, batch_first=True, padding_value=pad_idx)
    # Input: all but last, Target: all but first
    return batch_padded[:, :-1], batch_padded[:, 1:]

# Split data
train_size = int(0.9 * len(valid_smiles))
train_smiles = valid_smiles[:train_size]
val_smiles = valid_smiles[train_size:]

print(f"Train: {len(train_smiles):,} | Val: {len(val_smiles):,}")

# Create datasets
train_dataset = SMILESDataset(train_smiles, tokenizer)
val_dataset = SMILESDataset(val_smiles, tokenizer)

# Create dataloaders
batch_size = 32
pad_idx = tokenizer.token2idx[tokenizer.PAD]

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_idx), num_workers=0
)

print(f"✓ Dataloaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")


## 7. Transformer Architecture

Build a transformer decoder for autoregressive SMILES generation. Key components:
- **Positional Encoding**: Adds position information
- **Multi-Head Attention**: Captures dependencies
- **Causal Masking**: Prevents "seeing the future"


In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:x.size(1)]
        return self.dropout(x)


class SMILESTransformer(nn.Module):
    """Transformer for SMILES generation."""
    
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, 
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model, nhead, dim_feedforward, dropout, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        
        # Output
        self.fc_out = nn.Linear(d_model, vocab_size)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def generate_mask(self, sz):
        """Causal mask - prevents attending to future tokens."""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))
    
    def forward(self, src, tgt_mask=None):
        if tgt_mask is None:
            tgt_mask = self.generate_mask(src.size(1)).to(src.device)
        
        # Embed and encode
        src = self.embedding(src) * np.sqrt(self.d_model)
        src = self.pos_encoder(src)
        
        # Transform
        output = self.transformer(src, src, tgt_mask=tgt_mask)
        
        # Project to vocab
        return self.fc_out(output)

# Initialize model
model = SMILESTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=512,
    dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Model initialized:")
print(f"  Parameters: {total_params:,}")
print(f"  Size: ~{total_params * 4 / 1024**2:.1f} MB")


## 8. Training Configuration

Set up optimizer, loss function, and learning rate scheduler.


In [ ]:
# Training configuration
num_epochs = 10
learning_rate = 0.0001

# Loss (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print("⚙️ Training config:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Optimizer: Adam")
print(f"  Scheduler: CosineAnnealing")


## 9. Training Loop

Train the model using next-token prediction.


In [ ]:
history = {'train_loss': [], 'val_loss': []}

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for input_seq, target_seq in tqdm(loader, desc="Training"):
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)
        
        optimizer.zero_grad()
        output = model(input_seq)
        loss = criterion(output.reshape(-1, output.size(-1)), target_seq.reshape(-1))
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for input_seq, target_seq in loader:
            input_seq, target_seq = input_seq.to(device), target_seq.to(device)
            output = model(input_seq)
            loss = criterion(output.reshape(-1, output.size(-1)), target_seq.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(loader)

# Training loop
print("🚀 Starting training...\n")
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    print(f"\n📊 Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_smiles_model.pt')
        print(f"  ✓ Saved best model!")

print("\n✅ Training complete!")


In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))
epochs_range = range(1, len(history['train_loss']) + 1)
ax.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Progress', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f"\n📈 Final train loss: {history['train_loss'][-1]:.4f}")
print(f"📉 Best val loss: {best_val_loss:.4f}")


## 10. Molecule Generation

Use the trained model to generate novel SMILES strings!


In [ ]:
def generate_smiles(model, tokenizer, max_length=80, temperature=1.0, device='cpu'):
    """
    Generate a SMILES string using the trained model.
    
    Args:
        temperature: Controls randomness (higher = more random)
    """
    model.eval()
    
    # Start with START token
    start_idx = tokenizer.token2idx[tokenizer.START]
    sequence = [start_idx]
    
    with torch.no_grad():
        for _ in range(max_length):
            # Prepare input
            input_tensor = torch.tensor([sequence], dtype=torch.long).to(device)
            
            # Get predictions
            output = model(input_tensor)
            logits = output[0, -1, :] / temperature
            
            # Sample next token
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            
            # Stop if END token
            if next_token == tokenizer.token2idx[tokenizer.END]:
                break
            
            sequence.append(next_token)
    
    # Decode to SMILES
    tokens = [tokenizer.idx2token[idx] for idx in sequence[1:]]  # Skip START
    smiles = ''.join(tokens)
    return smiles

# Load best model
model.load_state_dict(torch.load('best_smiles_model.pt', map_location=device))
print("✓ Loaded best model\n")

# Generate molecules
print("🧪 Generating novel molecules...\n")
generated = []

for temp in [0.7, 1.0, 1.3]:
    print(f"Temperature {temp}:")
    for i in range(3):
        smi = generate_smiles(model, tokenizer, temperature=temp, device=device)
        generated.append(smi)
        print(f"  {i+1}. {smi}")
    print()

print(f"Generated {len(generated)} molecules")


## 11. Evaluation Metrics

Assess the quality of generated molecules using multiple metrics.


In [ ]:
# Generate more molecules for evaluation
print("Generating 100 molecules for evaluation...\n")
eval_molecules = []
for _ in tqdm(range(100)):
    smi = generate_smiles(model, tokenizer, temperature=1.0, device=device)
    eval_molecules.append(smi)

# Evaluation metrics
def evaluate_molecules(generated, training_set):
    """Calculate validity, uniqueness, and novelty."""
    results = {
        'total': len(generated),
        'valid': 0,
        'unique': 0,
        'novel': 0,
        'valid_smiles': [],
        'qed_scores': []
    }
    
    training_set = set(training_set)
    unique_smiles = set()
    
    for smi in generated:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            results['valid'] += 1
            canonical = Chem.MolToSmiles(mol)
            
            if canonical not in unique_smiles:
                results['unique'] += 1
                unique_smiles.add(canonical)
                results['valid_smiles'].append(canonical)
                results['qed_scores'].append(QED.qed(mol))
                
                if canonical not in training_set:
                    results['novel'] += 1
    
    # Calculate percentages
    results['validity_%'] = 100 * results['valid'] / results['total']
    results['uniqueness_%'] = 100 * results['unique'] / results['valid'] if results['valid'] > 0 else 0
    results['novelty_%'] = 100 * results['novel'] / results['unique'] if results['unique'] > 0 else 0
    
    return results

# Evaluate
results = evaluate_molecules(eval_molecules, train_smiles)

print("\n" + "="*50)
print("📊 Evaluation Results")
print("="*50)
print(f"\nTotal generated: {results['total']}")
print(f"Valid molecules: {results['valid']} ({results['validity_%']:.1f}%)")
print(f"Unique molecules: {results['unique']} ({results['uniqueness_%']:.1f}%)")
print(f"Novel molecules: {results['novel']} ({results['novelty_%']:.1f}%)")

if results['qed_scores']:
    print(f"\nDrug-likeness (QED):")
    print(f"  Mean: {np.mean(results['qed_scores']):.3f}")
    print(f"  Median: {np.median(results['qed_scores']):.3f}")

print("\n" + "="*50)


In [ ]:
# Visualize evaluation metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of metrics
categories = ['Validity', 'Uniqueness', 'Novelty']
values = [results['validity_%'], results['uniqueness_%'], results['novelty_%']]
colors = ['#2ecc71', '#3498db', '#e74c3c']

axes[0].bar(categories, values, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Percentage (%)', fontsize=12)
axes[0].set_title('Generation Quality Metrics', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[0].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

# QED distribution
if results['qed_scores']:
    axes[1].hist(results['qed_scores'], bins=20, color='mediumseagreen', 
                 edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('QED Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Drug-likeness Distribution', fontsize=14, fontweight='bold')
    axes[1].axvline(np.mean(results['qed_scores']), color='red', linestyle='--', 
                    linewidth=2, label=f"Mean: {np.mean(results['qed_scores']):.3f}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150)
plt.show()


## 12. Visualize Generated Molecules

Let's visualize the best generated molecules (highest QED scores).


In [ ]:
if results['valid_smiles']:
    # Sort by QED score
    sorted_indices = np.argsort(results['qed_scores'])[::-1]
    top_smiles = [results['valid_smiles'][i] for i in sorted_indices[:8]]
    top_qed = [results['qed_scores'][i] for i in sorted_indices[:8]]
    
    # Create molecules
    top_mols = [Chem.MolFromSmiles(smi) for smi in top_smiles]
    
    # Create legends
    legends = []
    for smi, qed in zip(top_smiles, top_qed):
        mol = Chem.MolFromSmiles(smi)
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        legends.append(f"QED: {qed:.3f}\\nMW: {mw:.1f}\\nLogP: {logp:.2f}")
    
    print("🏆 Top 8 Generated Molecules (by Drug-likeness)\\n")
    img = Draw.MolsToGridImage(
        top_mols,
        molsPerRow=4,
        subImgSize=(250, 250),
        legends=legends
    )
    display(img)
    
    # Save
    img.save('top_generated_molecules.png')
    print("\n💾 Saved to 'top_generated_molecules.png'")
else:
    print("⚠️ No valid molecules generated")


## 13. Summary and Conclusions

### 🎓 What We Accomplished

This notebook demonstrated a complete end-to-end pipeline for **de novo drug discovery using transformer models**:

#### ✅ Data Pipeline
- Automatic download of MOSES dataset (or fallback to curated molecules)
- SMILES validation and drug-likeness filtering
- No pre-existing data directories required!

#### ✅ Molecular Representation
- SMILES tokenization with custom regex patterns
- Vocabulary building from training data
- Token-to-index mapping for model input

#### ✅ Visualization
- RDKit-based 2D molecular structures
- Property calculation (MW, LogP, QED)
- Visual comparison of molecules

#### ✅ Model Architecture
- **Positional Encoding**: Sine/cosine embeddings for position
- **Multi-Head Self-Attention**: Captures molecular patterns
- **Causal Masking**: Enables autoregressive generation
- **Feed-Forward Networks**: Non-linear transformations

#### ✅ Training
- Next-token prediction objective
- Adam optimizer with learning rate scheduling
- Gradient clipping for stability
- Validation monitoring

#### ✅ Generation
- Temperature sampling for diversity control
- Autoregressive decoding
- Multiple sampling strategies

#### ✅ Evaluation
- **Validity**: Chemical validity via RDKit
- **Uniqueness**: Distinct generated molecules
- **Novelty**: Not in training set
- **Drug-likeness**: QED score distribution

### 🔬 Key Insights

1. **SMILES as Sequences**: Molecules can be represented as strings and learned like language
2. **Transformer Power**: Self-attention captures long-range dependencies in molecular structures
3. **Temperature Control**: Higher temperature = more diversity (but potentially lower quality)
4. **Evaluation Matters**: Multiple metrics needed to assess generative model quality

### 🚀 Next Steps & Extensions

#### Model Improvements
- **Larger Scale**: Train on full ChEMBL/ZINC databases (millions of molecules)
- **Bigger Models**: Increase layers, heads, and embedding dimensions
- **Longer Training**: More epochs for better convergence

#### Advanced Techniques
- **Conditional Generation**: Add property constraints (target MW, LogP, etc.)
- **Reinforcement Learning**: Optimize for specific properties using RL
- **Multi-objective**: Balance multiple properties simultaneously
- **Fragment-based**: Incorporate chemical fragment knowledge
- **3D Generation**: Extend to conformer/3D structure generation

#### Applications
- **Lead Optimization**: Generate analogs of known drugs
- **Scaffold Hopping**: Find alternative scaffolds with similar properties
- **Library Expansion**: Augment screening libraries
- **Property-driven Design**: Target specific pharmacological profiles

### 📚 Resources

**Papers:**
- Vaswani et al. (2017): \"Attention Is All You Need\"
- Polykovskiy et al. (2020): \"Molecular Sets (MOSES): A Benchmarking Platform\"
- Segler et al. (2018): \"Generating Focused Molecule Libraries\"
- Zhavoronkov et al. (2019): \"Deep learning enables rapid DDR1 kinase inhibitor identification\"

**Libraries:**
- RDKit: https://www.rdkit.org/
- PyTorch: https://pytorch.org/
- MOSES: https://github.com/molecularsets/moses

**Datasets:**
- ChEMBL: https://www.ebi.ac.uk/chembl/
- ZINC15: https://zinc15.docking.org/
- PubChem: https://pubchem.ncbi.nlm.nih.gov/

---

🎉 **Congratulations!** You've built a complete transformer-based molecular generation system from scratch!
